In [8]:
!pip install pandas requests


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path #دي بتساعدنا نتعامل مع: الملفات و الفولدرات و المسارات
import requests
import zipfile
import pandas as pd
import shutil

# مكان المشروع الحالي
BASE_DIR = Path.cwd()

print("Project folder:")
print(BASE_DIR)

Project folder:
d:\Courses\MachineLearning\Depi\Final Project


In [10]:
# Transportation folders

TRANSPORTATION_DIR = BASE_DIR / "data" / "transportation"

RAW_DIR = TRANSPORTATION_DIR / "raw"
EXTRACTED_DIR = TRANSPORTATION_DIR / "extracted"
PROCESSED_DIR = TRANSPORTATION_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Transportation folders created successfully!")
print()
print("RAW:", RAW_DIR)
print("EXTRACTED:", EXTRACTED_DIR)
print("PROCESSED:", PROCESSED_DIR)

Transportation folders created successfully!

RAW: d:\Courses\MachineLearning\Depi\Final Project\data\transportation\raw
EXTRACTED: d:\Courses\MachineLearning\Depi\Final Project\data\transportation\extracted
PROCESSED: d:\Courses\MachineLearning\Depi\Final Project\data\transportation\processed


In [11]:
# Download Cairo Transportation Data

GTFS_URL = "https://github.com/transportforcairo/GCR-Transit-Data/archive/refs/heads/master.zip"

ZIP_PATH = RAW_DIR / "GCR-Transit-Data.zip"

print("Downloading Cairo transportation data...")

response = requests.get(GTFS_URL, timeout=120)

response.raise_for_status()

with open(ZIP_PATH, "wb") as f:
    f.write(response.content)

print("Download completed successfully!")
print()
print("Saved at:")
print(ZIP_PATH)
print()
print("File size:",
      round(ZIP_PATH.stat().st_size / (1024 * 1024), 2),
      "MB")

Download completed successfully!

Saved at:
d:\Courses\MachineLearning\Depi\Final Project\data\transportation\raw\GCR-Transit-Data.zip

File size: 6.46 MB


In [12]:
# Extract downloaded data

print("Extracting transportation data...")

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACTED_DIR)

print("Extraction completed successfully!")

Extracting transportation data...
Extraction completed successfully!


In [13]:
# Find GTFS files

gtfs_files = list(EXTRACTED_DIR.rglob("*.txt"))

print("GTFS TXT files found:")
print("=" * 50)

for file in gtfs_files:
    print(file)

print()
print("Total TXT files:", len(gtfs_files))

GTFS TXT files found:
d:\Courses\MachineLearning\Depi\Final Project\data\transportation\extracted\GCR-Transit-Data-master\GTFS\20190729_GTFS_CTA-Paratransit\agency.txt
d:\Courses\MachineLearning\Depi\Final Project\data\transportation\extracted\GCR-Transit-Data-master\GTFS\20190729_GTFS_CTA-Paratransit\calendar.txt
d:\Courses\MachineLearning\Depi\Final Project\data\transportation\extracted\GCR-Transit-Data-master\GTFS\20190729_GTFS_CTA-Paratransit\calendar_dates.txt
d:\Courses\MachineLearning\Depi\Final Project\data\transportation\extracted\GCR-Transit-Data-master\GTFS\20190729_GTFS_CTA-Paratransit\feed_info.txt
d:\Courses\MachineLearning\Depi\Final Project\data\transportation\extracted\GCR-Transit-Data-master\GTFS\20190729_GTFS_CTA-Paratransit\frequencies.txt
d:\Courses\MachineLearning\Depi\Final Project\data\transportation\extracted\GCR-Transit-Data-master\GTFS\20190729_GTFS_CTA-Paratransit\routes.txt
d:\Courses\MachineLearning\Depi\Final Project\data\transportation\extracted\GCR-Tran

In [14]:
# Convert GTFS TXT files to CSV

processed_count = 0

for txt_file in gtfs_files:

    try:

        df = pd.read_csv(
            txt_file,
            low_memory=False
        )

        output_name = txt_file.name.replace(".txt", ".csv")

        output_path = PROCESSED_DIR / output_name

        df.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"✓ {txt_file.name}")
        print(f"  Rows: {len(df):,}")
        print(f"  Saved: {output_path}")
        print()

        processed_count += 1

    except Exception as e:

        print(f"✗ Error reading {txt_file.name}")
        print(" ", e)

print("=" * 50)
print(f"Converted files: {processed_count}")

✓ agency.txt
  Rows: 7
  Saved: d:\Courses\MachineLearning\Depi\Final Project\data\transportation\processed\agency.csv

✓ calendar.txt
  Rows: 1
  Saved: d:\Courses\MachineLearning\Depi\Final Project\data\transportation\processed\calendar.csv

✓ calendar_dates.txt
  Rows: 1
  Saved: d:\Courses\MachineLearning\Depi\Final Project\data\transportation\processed\calendar_dates.csv

✓ feed_info.txt
  Rows: 1
  Saved: d:\Courses\MachineLearning\Depi\Final Project\data\transportation\processed\feed_info.csv

✓ frequencies.txt
  Rows: 9,810
  Saved: d:\Courses\MachineLearning\Depi\Final Project\data\transportation\processed\frequencies.csv

✓ routes.txt
  Rows: 602
  Saved: d:\Courses\MachineLearning\Depi\Final Project\data\transportation\processed\routes.csv

✓ shapes.txt
  Rows: 181,717
  Saved: d:\Courses\MachineLearning\Depi\Final Project\data\transportation\processed\shapes.csv

✓ stops.txt
  Rows: 2,222
  Saved: d:\Courses\MachineLearning\Depi\Final Project\data\transportation\processed\s

In [15]:
# Check processed transportation data

csv_files = list(PROCESSED_DIR.glob("*.csv"))

print("Processed transportation files:")
print("=" * 50)

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)

    print(
        f"{file.name:25} "
        f"{len(df):,} rows"
    )

print()
print("Total CSV files:", len(csv_files))

Processed transportation files:
agency.csv                7 rows
calendar.csv              1 rows
calendar_dates.csv        1 rows
feed_info.csv             1 rows
frequencies.csv           9,810 rows
routes.csv                602 rows
shapes.csv                181,717 rows
stops.csv                 2,222 rows
stop_times.csv            222,804 rows
trips.csv                 9,810 rows

Total CSV files: 10


In [16]:
stops_path = PROCESSED_DIR / "stops.csv"

if stops_path.exists():

    stops = pd.read_csv(
        stops_path,
        low_memory=False
    )

    print("Stops dataset:")
    print(stops.shape)

    display(stops.head())

else:

    print("stops.csv was not found.")

Stops dataset:
(2222, 5)


,stop_name,stop_id,stop_desc,stop_lat,stop_lon
0,10th District (Al Hay Al Asher) - Nasr City,averts.hockey.woof,NaN,30.049050,31.378641
1,10th District (Al Hay Al Asher) - Nasr City,fermented.roosters.slacker,NaN,30.048353,31.379913
2,10th District (Al Hay Al Asher) - Nasr City,gravitate.singled.plus,NaN,30.049362,31.379839
3,10th District (Al Hay Al Asher) - Nasr City,iterative.sloping.down,NaN,30.048904,31.379050
4,10th District (Al Hay Al Asher) - Nasr City,repelled.face.stubble,NaN,30.049112,31.379221
